In [369]:
import pymor               
from pymor.basic import * 
import matplotlib.pyplot as plt

print(f"pyMOR is ready! Version: {pymor.__version__}")

pyMOR is ready! Version: 2025.1.2


In [370]:
# here we first create the problem
# this is the easiest problem


from pymor.analyticalproblems.domaindescriptions import LineDomain
from pymor.analyticalproblems.instationary import InstationaryProblem
from pymor.analyticalproblems.functions import ExpressionFunction
from pymor.analyticalproblems.functions import ConstantFunction


domain = LineDomain(domain=(0, 1), left='dirichlet', right='dirichlet')

diffusion = ConstantFunction(0.01, dim_domain=1) # fix parameter nu to 0.01

# The center of the Gaussian is now 'mu[0]'
rhs = ExpressionFunction("10 * exp(-((x[0] - mu[0])**2) / 0.01)", 
                         dim_domain=1, 
                         parameters={'mu': 1})

# Define the parameter space so the source stays inside the rod (0.1 to 0.9)
parameter_space = fom.parameters.space({'mu': (0.1, 0.9)})

initial_data = ConstantFunction(0.0, dim_domain=1)

problem_help = StationaryProblem(
    domain=domain,
    rhs=rhs,
    diffusion=diffusion,
)

problem = InstationaryProblem(
    stationary_part = problem_help,
    initial_data=initial_data,
    T=2.0,  # Final time: the simulation runs from t=0 to t=2.0
    name='Heat'
)


In [371]:
# here we discretize

fom, data = discretize_instationary_cg(problem, diameter=1/100, nt=50) # 50 time steps
print("FOM successfully created!")

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

FOM successfully created!


In [372]:
# tb example
from pymor.models.examples import thermal_block_example

fom_tb = thermal_block_example()

# create time-dep tb
from pymor.algorithms.timestepping import ImplicitEulerTimeStepper #check why this

fom_timetb = InstationaryModel(
    operator=fom_tb.operator,    
    rhs=fom_tb.rhs,              
    mass=fom_tb.products['l2'],               
    initial_data=fom_tb.operator.source.zeros(), 
    T=1.0,                         
    time_stepper=ImplicitEulerTimeStepper(nt=50),
    visualizer=fom_tb.visualizer
)

parameter_space_timetb = fom_timetb.parameters.space(0.1, 1)

# copy the products from tb
fom_timetb = fom_timetb.with_(products=fom_tb.products)


Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [373]:
# create std snapshots

sample_mus = parameter_space_timetb.sample_randomly(5) # 5 mus and we compute solution for every 50 timesteps

std_snapshots_timetb = fom_timetb.solution_space.empty()
for mu in sample_mus:

    U = fom_timetb.solve(mu)
    
    std_snapshots_timetb.append(U)

len(std_snapshots_timetb)

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

255

In [374]:
# create curly U

import numpy as np

U_curly_timetb = fom_timetb.solution_space.empty()      
   
# parameters
L = len(sample_mus) - 1  # parameter steps
M = fom_timetb.time_stepper.nt # time steps
N = (L + 1)*(M + 1)  # total number elements curly_U
dt = fom_timetb.T / M    # distant one time step
d_alpha = sample_mus[1]['diffusion'][0] - sample_mus[0]['diffusion'][0] # here i just took the difference w.r.t. the first (out of 4) parameter value


all_solutions = [fom_timetb.solve(mu) for mu in sample_mus]  # compute FOM solutions for all mu

for l in range(L + 1):
        
    vector = all_solutions[l][0].copy() # copy keeps all_solutions as it was
    vector *= np.sqrt(N) 
    U_curly_timetb.append(vector) # here we add starting values for every mu: u_h^{\alpha_l}(t_0^l) 

for j in range(1, M + 1):

    diff_t = (all_solutions[0][j] - all_solutions[0][j-1]) * (1.0 / dt)  
    diff_t *= np.sqrt(L + 1)
    U_curly_timetb.append(diff_t)  # here we add differeces w.r.t. starting time: D^t u_h^{\alpha_0}(t_j^0)

for l in range(1, L + 1):
    for j in range(1, M + 1):
            
        # (u_h^l(t_j^l) - u_h^{l-1}(t_j^{l-1})) over (\Delta \alpha)
        u_alpha_diff_j = (all_solutions[l][j] - all_solutions[l-1][j]) * (1.0 / d_alpha)  
        
        # (u_h^l(t_{j-1}^l) - u_h^{l-1}(t_{j-1}^{l-1})) over (\Delta \alpha)
        u_alpha_diff_j_minus_1 = (all_solutions[l][j-1] - all_solutions[l-1][j-1]) * (1.0 / d_alpha) 
        
        mixed_diff = (u_alpha_diff_j - u_alpha_diff_j_minus_1) * (1.0 / dt)  # take D^t
        U_curly_timetb.append(mixed_diff)  # here we add all other differences to U_curly

len(U_curly)
N

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

255

In [375]:
# mu = [0.1, 0.2, 0.5, 1]
# U = fom_timetb.solve(mu)
# fom_timetb.visualize(U);

In [376]:
# create curly U

# import numpy as np

# sample_mus_new = parameter_space.sample_uniformly(5)  # amount parameters 10 
# U_curly = fom.solution_space.empty()      # will be our final set of snapshots
   
# # Setup parameters
# L = len(sample_mus_new) - 1  # Number of parameter steps
# M = fom.time_stepper.nt # Number of time steps
# N = (L + 1) + M + (M * L)  # total number elements curly_U
# dt = fom.T / M    # distant one time step
# d_alpha = sample_mus_new[1]['mu'] - sample_mus_new[0]['mu'] # Assuming uniform spacing (not sure)


# all_solutions = [fom.solve(mu) for mu in sample_mus_new]  # compute FOM solutions for all mu

# for l in range(L + 1):
        
#     vector = all_solutions[l][0].copy() # copy keeps all_solutions as it was
#     vector *= np.sqrt(N) 
#     U_curly.append(vector) # here we add starting values for every mu: u_h^{\alpha_l}(t_0^l) 

# for j in range(1, M + 1):

#     diff_t = (all_solutions[0][j] - all_solutions[0][j-1]) * (1.0 / dt)  
#     diff_t *= np.sqrt(L + 1)
#     U_curly.append(diff_t)  # here we add differeces w.r.t. starting time: D^t u_h^{\alpha_0}(t_j^0)

# for l in range(1, L + 1):
#     for j in range(1, M + 1):
            
#         # (u_h^l(t_j^l) - u_h^{l-1}(t_j^{l-1})) over (\Delta \alpha)
#         u_alpha_diff_j = (all_solutions[l][j] - all_solutions[l-1][j]) * (1.0 / d_alpha)  
        
#         # (u_h^l(t_{j-1}^l) - u_h^{l-1}(t_{j-1}^{l-1})) over (\Delta \alpha)
#         u_alpha_diff_j_minus_1 = (all_solutions[l][j-1] - all_solutions[l-1][j-1]) * (1.0 / d_alpha) 
        
#         mixed_diff = (u_alpha_diff_j - u_alpha_diff_j_minus_1) * (1.0 / dt)  # take D^t
#         U_curly.append(mixed_diff)  # here we add all other differences to U_curly

# len(U_curly)
# N

In [377]:
# create std snapshots

# sample_mus = parameter_space.sample_randomly(5) # 5 mus and we compute solution for every 50 timesteps

# snapshots = fom.solution_space.empty()
# for mu in sample_mus:

#     U = fom.solve(mu)
    
#     snapshots.append(U)

# len(snapshots)

In [416]:
# POD with U_curly
r_curly_U = 5

from pymor.algorithms.pod import pod
U_pod_basis, U_pod_singular_values = pod(U_curly_timetb,
                                     product=fom_timetb.h1_0_semi_product,
                                     modes=r_curly_U
                                    )
len(U_pod_basis)

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

5

In [417]:
# POD with std 
r_std = 5

from pymor.algorithms.pod import pod
std_pod_basis, std_pod_singular_values = pod(std_snapshots_timetb,
                                     product=fom_timetb.products['h1_0_semi'],
                                     modes=r_std  
                                    )
len(std_pod_basis)

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

5

In [390]:
# construct ROM using curly U

from pymor.reductors.parabolic import ParabolicRBReductor

reductor_U = ParabolicRBReductor(
       fom=fom_timetb,
       RB=U_pod_basis,
       product=fom_timetb.h1_0_semi_product,
       coercivity_estimator=None,
       check_orthonormality=None,
       check_tol=None
   )
rom_U_timetb = reductor_U.reduce()

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [391]:
# construct ROM using std

from pymor.reductors.parabolic import ParabolicRBReductor

reductor_std = ParabolicRBReductor(
       fom=fom_timetb,
       RB=std_pod_basis,
       product=fom_timetb.h1_0_semi_product,
       coercivity_estimator=None,
       check_orthonormality=None,
       check_tol=None
   )
rom_std_timetb = reductor_std.reduce()

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [403]:
# test my ROMs (if its u_..., we already have a specific parameter)

test_mu = parameter_space_timetb.sample_randomly(1)[0]

u_rom_std = rom_std_timetb.solve(test_mu) # solve rom for this mu in low dim
u_rec_std = reductor_std.reconstruct(u_rom_std) # back to high dim

u_rom_U = rom_U_timetb.solve(test_mu) # solve rom for this mu in low dim
u_rec_U = reductor_U.reconstruct(u_rom_U) # back to high dim

u_fom_timetb = fom_timetb.solve(test_mu)

fom_timetb.visualize((u_fom_timetb, u_rec_std, u_rec_U), 
                     legend=("Full Order", "Standard ROM", "Curly U ROM"));

Accordion(children=(HTML(value='', layout=Layout(height='16em', width='100%')),), titles=('Log Output',))

In [397]:
print(f"Basis length: {len(U_pod_basis)}")
print(f"Coeffs shape: {coeffs.shape}")

Basis length: 5
Coeffs shape: (5, 51)


In [424]:
# for the error anylysis: (u_h - u_r) = (u_h - Pr(u_h)) + (Pr(u_h) - u_r))
# compute Pr(u_h), the projection of the FOM solution onto U_pod_basis

coeffs = fom_timetb.products['h1_0_semi'].apply2(U_pod_basis, u_fom) 

u_projected = U_pod_basis.lincomb(coeffs)

error_proj = (u_fom_timetb - u_projected).norm(fom_timetb.products['h1_0_semi']) # (u_h - Pr(u_h))

error_rom_U = (u_fom_timetb - u_rec_U).norm(fom_timetb.products['h1_0_semi'])  # (u_h - u_r)

error_rom_std = (u_fom_timetb - u_rec_std).norm(fom_timetb.products['h1_0_semi'])

# for error analysis we also wanna set the eigenvalue tail

# we start with the std case
energies_std = std_pod_singular_values**2

total_energy = np.sum(energies_std) # the whole sum
# To get the true TOTAL energy of all snapshots
captured_energy = np.sum(energies_std[:r_std])
captured_energy, total_energy
# # 4. The Tail (Sigma_r)
# sigma_r = 1 - (captured_energy / total_energy)

# print(f"Sigma_r for r={r}: {sigma_r:.2e}")

(np.float64(36.490175446298664), np.float64(36.510634265409124))

In [408]:
# visualisation of u_h, Pr(u_h), u_N (obtained std ROM), u_N (obtained curlyU ROM) 

fom_timetb.visualize(
    (u_fom, u_projected, u_rec_U, u_rec_std), 
    legend=("Full Order", "Best Possible (Proj)", "Curly U ROM", "Standard ROM"),
);

In [412]:
# visualisation of the errors

diff_proj = u_fom_timetb - u_projected
diff_rom_U = u_fom_timetb - u_rec_U
diff_rom_std = u_fom_timetb - u_rec_std


fom_timetb.visualize(
    (diff_rom_U, diff_rom_std, diff_proj),
    legend=("Curly U Error Field", "Standard ROM Error Field", "Projection Error Field"),
    separate_colorbars=True
)


In [411]:
print(f"ROM U steps: {len(error_rom_U)}")
print(f"ROM Std steps: {len(error_rom_std)}")
print(f"Proj steps: {len(error_proj)}")

ROM U steps: 51
ROM Std steps: 51
Proj steps: 51


In [406]:
# fom.visualize((u_fom, u_rec_std, u_rec_U), 
#               legend=('FOM', 'ROM std', 'ROM curly U'))

In [385]:
# here we see the errors are very close, but not similar
err_std = u_fom - u_rec_std
err_curly = u_fom - u_rec_U

err_std;
err_curly;


In [386]:
# just to check if compareable
print(f"FOM steps:    {len(u_fom)}")
print(f"STD ROM steps: {len(u_rec_std)}")
print(f"Curly ROM steps: {len(u_rec_U)}")

print(f"FOM grid dim:    {u_fom.dim}")
print(f"STD ROM grid dim: {u_rec_std.dim}")
print(f"Curly ROM grid dim: {u_rec_U.dim}")

FOM steps:    51
STD ROM steps: 51
Curly ROM steps: 51
FOM grid dim:    20201
STD ROM grid dim: 20201
Curly ROM grid dim: 20201
